In [1]:
import pandas as pd
import numpy as np
import joblib
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix

pd.set_option('display.max_columns', None)

In [2]:
df = pd.read_csv('../data/german_credit_clean.csv')
print(df.shape)

(1000, 22)


In [3]:
# 'sex' is kept out of the model features; it is only used later for the fairness audit
X = df.drop(columns=['default', 'sex'])
y = df['default']
sex_full = df['sex']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

sex_train = sex_full.loc[X_train.index]
sex_test = sex_full.loc[X_test.index]

print(X_train.shape, X_test.shape)
print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

(800, 20) (200, 20)
default
0    0.7
1    0.3
Name: proportion, dtype: float64
default
0    0.7
1    0.3
Name: proportion, dtype: float64


In [4]:
categorical_cols = ['checking_account', 'credit_history', 'purpose', 'savings_account',
                    'employment_since', 'personal_status_sex', 'other_debtors', 'property',
                    'other_installment_plans', 'housing', 'job', 'telephone', 'foreign_worker']

X_train_enc = pd.get_dummies(X_train, columns=categorical_cols, drop_first=True)
X_test_enc = pd.get_dummies(X_test, columns=categorical_cols, drop_first=True)

# Guard against a category appearing in only one split
X_train_enc, X_test_enc = X_train_enc.align(X_test_enc, join='left', axis=1, fill_value=0)

print(X_train_enc.shape, X_test_enc.shape)

(800, 48) (200, 48)


In [5]:
X_train_sm = sm.add_constant(X_train_enc.astype(float))
X_test_sm = sm.add_constant(X_test_enc.astype(float), has_constant='add')

logit_result = sm.Logit(y_train, X_train_sm).fit(maxiter=200)
print(logit_result.summary())

Optimization terminated successfully.
         Current function value: 0.445661
         Iterations 7
                           Logit Regression Results                           
Dep. Variable:                default   No. Observations:                  800
Model:                          Logit   Df Residuals:                      751
Method:                           MLE   Df Model:                           48
Date:                Mon, 23 Nov 2026   Pseudo R-squ.:                  0.2704
Time:                        03:16:44   Log-Likelihood:                -356.53
converged:                       True   LL-Null:                       -488.69
Covariance Type:            nonrobust   LLR p-value:                 1.142e-31
                                                    coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------------------
const                                 

In [6]:
odds_ratios = pd.DataFrame({
    'coef': logit_result.params,
    'odds_ratio': np.exp(logit_result.params),
    'p_value': logit_result.pvalues
})
odds_ratios_sig = odds_ratios[odds_ratios['p_value'] < 0.05].sort_values('odds_ratio', ascending=False)
odds_ratios_sig

,coef,odds_ratio,p_value
housing_rent,1.531358,4.624451,0.007069
foreign_worker_yes,1.465734,4.330722,0.037526
purpose_education,1.192855,3.296479,0.021805
property_unknown/no property,1.177006,3.244645,0.015816
installment_rate,0.361187,1.435032,0.000246
duration,0.028172,1.028573,0.010683
credit_amount,0.000133,1.000133,0.016916
personal_status_sex_male: single,-0.611638,0.542462,0.009627
savings_account_unknown/no savings,-0.806178,0.446562,0.037856
other_installment_plans_none,-0.888494,0.411275,0.000976


In [7]:
y_pred_proba = logit_result.predict(X_test_sm)
y_pred = (y_pred_proba >= 0.5).astype(int)

print("ROC-AUC:", roc_auc_score(y_test, y_pred_proba))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

ROC-AUC: 0.7988095238095237

Classification Report:
               precision    recall  f1-score   support

           0       0.83      0.86      0.85       140
           1       0.64      0.60      0.62        60

    accuracy                           0.78       200
   macro avg       0.74      0.73      0.73       200
weighted avg       0.78      0.78      0.78       200


Confusion Matrix:
 [[120  20]
 [ 24  36]]


In [8]:
saved = joblib.load('../src/logit_final_model.pkl')
print("Coefficients match saved model:", np.allclose(saved.params.values, logit_result.params.values))

Coefficients match saved model: True
